### Exercise 1.1 - DBMS Properties: Applied Reasoning

Below are four desired properties of a DBMS. For **each one**:
1. Write a one-sentence explanation **in your own words**
2. Give a **concrete example from the Nobel Prize dataset** we use in this lab - http://api.nobelprize.org/v1/prize.json (not a generic scenario)
3. Describe **what would go wrong** in our Nobel Prize database if that property were violated

```
(a) Integration
(b) Independence
(c) Isolation
(d) Interface
```

*Hint: Think about what our `winners` table actually stores and how it might be used by different parts of an application.*

*(Your answer here - be specific to the Nobel Prize context)*
(a) Integration
Integration means the database keeps related information together in one system so it is easier to store, connect, and look up data. In this Nobel Prize database, the winners table holds things like the year, category, winner name, motivation, and prize share in one place. That makes it easy to search for winners and compare records. If integration did not exist, the data could be scattered across different places, which would make it harder to manage and could lead to missing or inconsistent information.

(b) Independence
Independence means the way the data is stored can change without forcing users to completely change how they access it. For example, the Nobel Prize data could be reorganized into separate tables later, but users should still be able to run similar queries. If this property did not exist, even a small change to the database structure could break old queries and make the system harder to maintain.

(c) Isolation
Isolation means different database actions can happen at the same time without interfering with each other. For example, if one person is updating Nobel Prize records while someone else is searching for winners, the search should not show incomplete or half-updated data. Without isolation, users could see data in the middle of being changed, which could cause confusion or wrong results.

(d) Interface
Interface means the database gives users a way to work with the data, usually through a language like SQL. In this lab, the interface is what lets us ask questions about the Nobel Prize winners instead of reading through all the raw data ourselves. Without a good interface, using the database would be much more difficult because people would have to dig through the raw file manually.

### Exercise 1.2 - Debugging SQL: Explain the Bug

A classmate wrote the following SQL query intending to count the number of unique first names in the `students` table:

```sql
SELECT DISTINCT(COUNT(first_name)) FROM students;
```

**Before running anything**, answer:

(a) What do you think this query will actually return? Walk through what SQL does step-by-step with `DISTINCT` and `COUNT` applied this way.

(b) Why is this query logically wrong - what is the conceptual misunderstanding?

(c) Write the corrected query.

(d) Are there cases where `DISTINCT` and `COUNT` can be correctly combined? Give an example.

<b>Write your answers below</b> <br>
*(a) Step-by-step prediction:* This query will return one number, not the number of unique first names. SQL will first do COUNT(first_name), which counts how many rows have a non-null first_name. After that, DISTINCT is being applied to that single result, so it does basically nothing. Since there is only one count value coming back, the query just returns that one total.

*(b) Conceptual misunderstanding:* The problem is that the classmate is mixing up counting rows with counting distinct values. COUNT(first_name) counts every non-null first name in the table, even if the same name appears many times. If the goal is to count unique first names, DISTINCT has to be applied to the first_name values first, not to the result of the count.

*(c) Corrected query:* The correct query is:
SELECT COUNT(DISTINCT first_name) FROM students;

*(d) Valid combined use:* Yes, DISTINCT and COUNT can be used together the right way. A good example is:
SELECT COUNT(DISTINCT category) FROM winners;
That would count how many different prize categories appear in the winners table. The key idea is that DISTINCT should go inside the COUNT when you want to count unique values.

### Exercise 1.3 - Spot the Syntax Error and Predict the Impact

```sql
UPDATE students SET first_name='Bob' WHERE first_name='Robert'. 
```

(a) Identify the syntax error. Will this query run or fail?

(b) If the error were removed and the query ran successfully - **how many rows would be updated?** Would it affect only one student named Robert, or all of them? Explain why.

(c) If you only wanted to rename **one specific Robert** (say, `student_id = 1042`), write the corrected query.

(d) What real-world consequence could bulk `UPDATE` operations like this have in a production database? Give a scenario where this could cause serious problems.

<b>Write your answers below</b> <br>
*(a)* The syntax error is the period at the end of the query. SQL statements should end with a semicolon, not a dot. So as written, the query would fail instead of running.

*(b)* If you remove the error and the query runs, it would update every row where first_name = 'Robert', not just one person. That is because the WHERE clause is only checking the first name, so anyone in the table named Robert would get changed to Bob.

*(c)* 
UPDATE students
SET first_name = 'Bob'
WHERE student_id = 1042;

*(d)* In a real database, a bulk UPDATE like this could cause a lot of problems if it changes more rows than expected. For example, if a school database changed every student named Robert to Bob by mistake, records could become inaccurate, reports could be wrong, and it might be hard to tell which names were supposed to be changed and which were not.

### Exercise 1.4 - Read, Predict, Fix, Generalize

```sql
SELECT 5* FROM winners;
```

(a) What was the student *trying* to do, and what is wrong with this query?

(b) Write the correct query that shows only the first 5 rows.

(c) SQL does not have universal syntax for row limiting across all databases. What is the equivalent of `LIMIT 5` in **Microsoft SQL Server** and in **Oracle SQL**? Why do you think these differ across systems?

(d) In what situation might limiting query results actually **hide an important problem** in your data?

<b>Write your answers below</b> <br>
*(a)* The student was probably trying to show only the first 5 rows from the winners table. What is wrong is the syntax: SELECT 5* FROM winners; is not valid SQL. You cannot put the 5 there like that. To limit rows, you need something like LIMIT 5 or the database’s own version of row limiting.

*(b)*
SELECT *
FROM winners
LIMIT 5;

*(c)* Microsoft SQL Server:
SELECT TOP 5 *
FROM winners;

In Oracle SQL:
SELECT *
FROM winners
FETCH FIRST 5 ROWS ONLY;

They differ because SQL is standardized only to a point, and different database systems were developed by different companies over time. Each one added its own syntax and features before newer standards were adopted.

*(d)* Limiting results can hide problems when the bad data is not in the first few rows. For example, if the first 5 rows in winners look fine, you might think the table is clean, but there could still be missing names, duplicate rows, or wrong years later in the table that you never see because of the limit.

## Problem 2 - Nobel Prize Winners Database

The Nobel Prize website provides comprehensive information on prizes awarded annually in Physics, Chemistry, Medicine, Literature, Peace, and Economic Sciences.

We will build a database for Nobel Prize winner data from http://api.nobelprize.org/v1/prize.json

**Before writing any code**, answer Exercise 2.0 below.

### Exercise 2.0 - Design Before You Explore (Answer BEFORE running any code)

Based on your general knowledge of how Nobel Prizes work:

(a) Predict: what fields (columns) do you expect to find in this JSON dataset? List at least 5.

(b) Predict: will this be a flat structure (a simple list of rows) or nested (e.g., a prize containing multiple laureates)? Explain your reasoning.

(c) Predict: what challenges might arise when loading this nested JSON into a flat database table?

*(You will check your predictions in Tasks 2.1 and 2.2.)*

<b>Write your answers below</b> <br>
*(a)* Before looking at the code, I would expect this JSON dataset to include fields like year, category, laureate id, first name, last name, motivation/reason for the award, and maybe share if a prize is split between multiple winners. I would also expect that some kind of overall prize description or prize-level information could appear too.

*(b)* I would expect the structure to be nested, not completely flat. The main reason is that one Nobel Prize category in a given year can have more than one winner, so it makes sense that the dataset would store one prize entry and then group multiple laureates inside it. That seems more natural than repeating the whole prize information over and over in one simple list.

*(c)* One challenge with loading that kind of nested JSON into a flat database table is that a single prize can have multiple laureates, so you have to decide whether to repeat the prize information for each winner or split the data into separate tables. Another issue is that some winners might be organizations instead of individual people, which could make name fields inconsistent. There could also be missing fields, different formats across records, or extra nested parts that do not fit neatly into one table row.

### <font color="grey">Task 2.1: Structure of a JSON File</font>

##### <font color="grey">Read the json data from http://api.nobelprize.org/v1/prize.json into a dictionary called `nobels` and find the main keys in the dictionary</font>

In [1]:
import requests

url = "https://api.nobelprize.org/v1/prize.json"
headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json"
}

response = requests.get(url, headers=headers)
print(response.status_code)

nobels = response.json()

200


In [2]:
nobels.keys()

dict_keys(['prizes'])

##### Print the first json record

In [3]:
# your code
nobels["prizes"][0]

{'year': '2025',
 'category': 'chemistry',
 'laureates': [{'id': '1053',
   'firstname': 'Susumu',
   'surname': 'Kitagawa',
   'motivation': '"for the development of metal–organic frameworks"',
   'share': '3'},
  {'id': '1054',
   'firstname': 'Richard',
   'surname': 'Robson',
   'motivation': '"for the development of metal–organic frameworks"',
   'share': '3'},
  {'id': '1055',
   'firstname': 'Omar M.',
   'surname': 'Yaghi',
   'motivation': '"for the development of metal–organic frameworks"',
   'share': '3'}]}

### Reflection 2.1 - Check Your Predictions

Look back at your answers in Exercise 2.0.

(a) Which of your predicted fields were correct? Which were wrong or missing?

(b) Was the structure flat or nested? Did this match your prediction? What does this tell you about how API data is typically structured?

<b>Write your answers below</b> <br>
*(a)* Some of my predictions were correct. I expected fields like year, category, firstname, surname, motivation, and share, and those all showed up in the data. One thing I did not fully predict was that the laureate information would be stored inside a nested laureates list instead of everything being in one flat row.

*(b)* The structure was nested, not flat, and that did match my prediction. Each prize record has general information like the year and category, and then the winners are grouped inside the laureates list. This shows that API data is usually organized by grouping related data together instead of flattening it right away.

### <font color="grey">Task 2.2: Inner json structure</font>

##### <font color="grey">Each entry in nobels['prizes'] is a dictionary of year + category + list of laureates. Print the keys within each record.</font>

In [4]:
# your code
for record in nobels["prizes"]:
    print(record.keys())


dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'overallMotivation', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'laureates'])
dict_keys(['year', 'category', 'la

### <font color="grey">Print the record with index 1 from nobels['prizes']</font>

In [5]:
# your code
nobels["prizes"][1]


{'year': '2025',
 'category': 'economics',
 'overallMotivation': '"for having explained innovation-driven economic growth"',
 'laureates': [{'id': '1058',
   'firstname': 'Joel',
   'surname': 'Mokyr',
   'motivation': '"for having identified the prerequisites for sustained growth through technological progress"',
   'share': '2'},
  {'id': '1059',
   'firstname': 'Philippe',
   'surname': 'Aghion',
   'motivation': '"for the theory of sustained growth through creative destruction"',
   'share': '4'},
  {'id': '1060',
   'firstname': 'Peter',
   'surname': 'Howitt',
   'motivation': '"for the theory of sustained growth through creative destruction"',
   'share': '4'}]}

### <font color="grey">Print the most recent Nobel laureate</font>

In [6]:
# your code
nobels["prizes"][0]["laureates"][0]


{'id': '1053',
 'firstname': 'Susumu',
 'surname': 'Kitagawa',
 'motivation': '"for the development of metal–organic frameworks"',
 'share': '3'}

### Reasoning Question 2.2 - Index Assumptions

You accessed the "most recent" laureate using an index position.

(a) What assumption are you making about the ordering of records in the JSON file? Is this assumption safe to rely on?

(b) How would you write more **robust** code that finds the most recent laureate by actually checking the year field, rather than relying on index position? Write that version.

<b>Write your answers below</b> <br>
*(a)* The assumption is that the JSON records are already ordered from most recent to oldest, so using index 0 means I am treating the first prize entry as the newest one. That is not completely safe to rely on because the order could change, and APIs do not always guarantee that records will stay in the same order.

*(b)* A more robust way is to actually check the year field and find the largest year first, instead of assuming the first record is the newest.

In [7]:
# Robust version that checks the year field
latest_prize = max(
    [record for record in nobels["prizes"] if "laureates" in record],
    key=lambda record: int(record["year"])
)

print("Most recent year:", latest_prize["year"])
print(latest_prize["laureates"][0])


Most recent year: 2025
{'id': '1053', 'firstname': 'Susumu', 'surname': 'Kitagawa', 'motivation': '"for the development of metal–organic frameworks"', 'share': '3'}


### <font color="grey">Task 2.3: Memory Requirements for Database</font>

We want to design a database storing **Year, Category, First Name, Last Name, Motivation, and Share** for each Nobel laureate.

**Before writing code**, fill in the prediction table below. Then run code to measure the actual values and compare.

**Predictions (complete BEFORE running any code):**

| Field | Predicted SQL Type | Predicted Max Length/Size | Reasoning |
|-------|-------------------|--------------------------|----------|
| Year | SMALLINT | 4 | years are 4 digits |
| Category | VARCHAR | 10 | categories are short words |
| First Name | VARCHAR | 60 | some entries are organization names |
| Last Name | VARCHAR | 30 | surnames can have spaces or multiple parts |
| Motivation | TEXT | 400 | motivation can be long |
| Share | TINYINT | 1 | values are small digits like 1, 2, 3, 4 |

In [8]:
# your code - compute actual max sizes for each field and identify years with no laureates
max_year = 0
max_category = 0
max_first = 0
max_last = 0
max_motivation = 0
max_share = 0

years_with_no_laureates = []

for prize in nobels["prizes"]:
    max_year = max(max_year, len(prize.get("year", "")))
    max_category = max(max_category, len(prize.get("category", "")))

    if "laureates" not in prize:
        years_with_no_laureates.append((prize["year"], prize["category"]))
    else:
        for laureate in prize["laureates"]:
            max_first = max(max_first, len(laureate.get("firstname", "")))
            max_last = max(max_last, len(laureate.get("surname", "")))
            max_motivation = max(max_motivation, len(laureate.get("motivation", "")))
            max_share = max(max_share, len(laureate.get("share", "")))

print("Actual max lengths:")
print("Year:", max_year)
print("Category:", max_category)
print("First Name:", max_first)
print("Last Name:", max_last)
print("Motivation:", max_motivation)
print("Share:", max_share)

print("\nYears/categories with no laureates:")
for item in years_with_no_laureates:
    print(item)


Actual max lengths:
Year: 4
Category: 10
First Name: 59
Last Name: 26
Motivation: 376
Share: 1

Years/categories with no laureates:
('1972', 'peace')
('1967', 'peace')
('1966', 'peace')
('1956', 'peace')
('1955', 'peace')
('1948', 'peace')
('1943', 'literature')
('1943', 'peace')
('1942', 'chemistry')
('1942', 'literature')
('1942', 'peace')
('1942', 'physics')
('1942', 'medicine')
('1941', 'chemistry')
('1941', 'literature')
('1941', 'peace')
('1941', 'physics')
('1941', 'medicine')
('1940', 'chemistry')
('1940', 'literature')
('1940', 'peace')
('1940', 'physics')
('1940', 'medicine')
('1939', 'peace')
('1935', 'literature')
('1934', 'physics')
('1933', 'chemistry')
('1932', 'peace')
('1931', 'physics')
('1928', 'peace')
('1925', 'medicine')
('1924', 'chemistry')
('1924', 'peace')
('1923', 'peace')
('1921', 'medicine')
('1919', 'chemistry')
('1918', 'literature')
('1918', 'peace')
('1918', 'medicine')
('1917', 'chemistry')
('1917', 'medicine')
('1916', 'chemistry')
('1916', 'peace')
(

#### Post-Measurement Reflection

(a) Compare the actual measurements to your predictions. Where were you wrong, and why?

(b) Finalize your SQL data type choices based on actual measurements:

| Field | Final SQL Type | Changed from prediction? |
|-------|---------------|--------------------------|
| Year | | |
| Category | | |
| First Name | | |
| Last Name | | |
| Motivation | | |
| Share | | |

(c) You found years with no laureates. Look up one real example - why did that year have no Nobel Prize awarded? How does this edge case affect your schema design?

(d) Why might it be dangerous to set `VARCHAR` sizes exactly at the measured maximum, rather than adding a buffer?

<b>Write your answers below</b> <br>
*(a)* Most of my predictions were close, but a few fields needed more room than I first expected. Year and Share were straightforward and stayed small. Category was still short, but the longest value was 10 characters. First Name turned out to need a lot more space because some records are organizations instead of people, so the longest first-name value was much longer than a normal person’s name. Motivation was also much longer than a normal VARCHAR, so using TEXT makes more sense there. Based on the dataset, the actual maximum lengths are 4 for Year, 10 for Category, 59 for First Name, 26 for Last Name, 376 for Motivation, and 1 for Share.

*(b) see table above*
| Field      | Final SQL Type | Changed from prediction? |
| ---------- | -------------- | ------------------------ |
| Year       | `SMALLINT`     | No                       |
| Category   | `VARCHAR(20)`  | Yes                      |
| First Name | `VARCHAR(80)`  | Yes                      |
| Last Name  | `VARCHAR(40)`  | Yes                      |
| Motivation | `TEXT`         | No                       |
| Share      | `TINYINT`      | No                       |


*(c)* A real example is the Nobel Prize in Physics in 1942, when no prize was awarded. This happened during World War II, when several Nobel Prizes were not given out. For database design, that means a prize entry should still be able to exist even if there are no laureates linked to it. The best way to handle that is by keeping Prizes and Laureates in separate tables, so a prize can have no winners, one winner, or multiple winners. 

*(d)* It is risky to set VARCHAR sizes exactly to the current maximum because future data might be slightly longer. If that happens, inserts could fail or values could get cut off. Adding a small buffer makes the schema more flexible, especially for names and organization titles.

## Problem 3 - Creating the Database

Now it is time to store the data in a permanent database.

### <font color="grey">Task 3.1: Initialize</font>

Initialize a database called `nobels.db`

In [9]:
# your code
import sqlite3

conn = sqlite3.connect("nobels.db")
cur = conn.cursor()

print("nobels.db initialized")


nobels.db initialized


---

### <font color="grey">Task 3.2: Winners Table</font>

Create a table called **winners** in the database with columns: year, category, first name, last name, motivation, and share - using the appropriate data types you finalized in Task 2.3.

In [10]:
# your code
cur.execute("DROP TABLE IF EXISTS winners")

cur.execute("""
CREATE TABLE winners (
    year SMALLINT,
    category VARCHAR(20),
    firstname VARCHAR(80),
    surname VARCHAR(40),
    motivation TEXT,
    share TINYINT
)
""")

conn.commit()
print("winners table created")


winners table created


##### Show winners Table info

In [11]:
# your code
cur.execute("PRAGMA table_info(winners)")
cur.fetchall()


[(0, 'year', 'SMALLINT', 0, None, 0),
 (1, 'category', 'VARCHAR(20)', 0, None, 0),
 (2, 'firstname', 'VARCHAR(80)', 0, None, 0),
 (3, 'surname', 'VARCHAR(40)', 0, None, 0),
 (4, 'motivation', 'TEXT', 0, None, 0),
 (5, 'share', 'TINYINT', 0, None, 0)]

##### List Column Names

In [12]:
# your code
cur.execute("PRAGMA table_info(winners)")
columns = [col[1] for col in cur.fetchall()]
columns


['year', 'category', 'firstname', 'surname', 'motivation', 'share']

### Exercise 3.2 - Primary Key Design Decision

The original lab stated: *"We could do a combination of year and category as unique, since no two rows can have the same combination of year and category."*

(a) **Challenge this claim**: Is `(year, category)` truly a unique combination per row? Can you identify a counterexample from this actual dataset where two rows share the same year and category?

(b) Propose a better primary key strategy for this table. Justify your choice.

(c) What is a **surrogate key**? Would it be appropriate here, and why?

<b>Write your answers below</b> <br>
*(a)* No, (year, category) is not unique for each row in this dataset. A single prize can have multiple laureates, so several rows can share the same year and category. For example, the 2025 chemistry prize has multiple laureates, so those rows would all have the same year and category.

*(b)* A better primary key would be a separate id column, like winner_id INTEGER PRIMARY KEY AUTOINCREMENT. That works better because each row gets its own unique value, even when multiple winners share the same year and category.

*(c)* A surrogate key is an artificial key added just to uniquely identify each row, instead of using existing data fields. Yes, it would be appropriate here because the natural fields in this table do not always make each row unique.

### <font color="grey">Task 3.3: Load the data into winners table</font>

In [13]:
# your code
rows = []

for prize in nobels["prizes"]:
    if "laureates" in prize:
        for laureate in prize["laureates"]:
            rows.append((
                int(prize["year"]),
                prize["category"],
                laureate.get("firstname", ""),
                laureate.get("surname", ""),
                laureate.get("motivation", ""),
                int(laureate.get("share", 0))
            ))

cur.executemany("""
INSERT INTO winners (year, category, firstname, surname, motivation, share)
VALUES (?, ?, ?, ?, ?, ?)
""", rows)

conn.commit()
print("Inserted", len(rows), "rows into winners")


Inserted 1026 rows into winners


### <font color="grey">Task 3.4: Querying the winners Table</font>

Reference: https://dev.mysql.com/doc/refman/8.0/en/select.html

For **each query below**, you must:
1. Write your **prediction** before running
2. Write and run the query
3. Note if your prediction was correct

### <font color="grey">Task 3.4.1: How many entries are in the table?</font>

**Prediction (write before running):** How many rows do you expect? Nobel Prizes have been awarded since 1901 in 6 categories, and prizes can be shared. Make an educated estimate and show your reasoning.

<b>Write your answers below</b> <br>
*(Your estimate and reasoning - before running the query)* <br>

I would guess the table has around 900 to 1,100 rows. 
My reasoning is that Nobel Prizes have been awarded for many years across several categories, and many prizes are shared by multiple laureates. 
Since this table stores one row per laureate, the total should be much higher than just the number of prize records.

In [14]:
# your code
cur.execute("SELECT COUNT(*) FROM winners")
cur.fetchone()[0]


1026

**Post-run reflection:** Was your prediction close? What factors did you over- or under-estimate?

<b>Write your answers below</b> <br>
*(Your reflection)* <br>

My prediction was pretty close because the actual total was 1026 rows, which falls in the range I guessed. I expected the count to be high since the table stores one row per laureate, not one row per prize, and many Nobel Prizes are shared by multiple winners. One thing that keeps the total from being even higher is that some years or categories had no laureates.

### <font color="grey">Task 3.4.2: Show the first 5 rows in the table</font>

In [15]:
# your code
cur.execute("SELECT * FROM winners LIMIT 5")
cur.fetchall()

[(2025,
  'chemistry',
  'Susumu',
  'Kitagawa',
  '"for the development of metal–organic frameworks"',
  3),
 (2025,
  'chemistry',
  'Richard',
  'Robson',
  '"for the development of metal–organic frameworks"',
  3),
 (2025,
  'chemistry',
  'Omar M.',
  'Yaghi',
  '"for the development of metal–organic frameworks"',
  3),
 (2025,
  'economics',
  'Joel',
  'Mokyr',
  '"for having identified the prerequisites for sustained growth through technological progress"',
  2),
 (2025,
  'economics',
  'Philippe',
  'Aghion',
  '"for the theory of sustained growth through creative destruction"',
  4)]

### <font color="grey">Task 3.4.3: Show the winners for 2024</font>

**Prediction:** List the 2024 Nobel Prize winners you know from memory or general knowledge. Then verify.

Show records in this format:
```
year: 2024
category: Chemistry
fname: David
lname: Baker
motivation: for computational protein design
share: 3
```

<b>Write your answers below</b> <br>
*(Your prediction of 2024 winners - write before running)* <br>
I do not know the 2024 winners from memory, but I expect this query to return multiple rows. 
My reasoning is that Nobel Prizes are awarded across several categories each year, and some categories have more than one winner.

In [16]:
# your code
cur.execute("""
SELECT year, category, firstname, surname, motivation, share
FROM winners
WHERE year = 2024
ORDER BY category, surname
""")

rows = cur.fetchall()

for row in rows:
    print(f"year: {row[0]}")
    print(f"category: {row[1].title()}")
    print(f"fname: {row[2]}")
    print(f"lname: {row[3]}")
    print(f"motivation: {row[4]}")
    print(f"share: {row[5]}")
    print()


year: 2024
category: Chemistry
fname: David
lname: Baker
motivation: "for computational protein design"
share: 2

year: 2024
category: Chemistry
fname: Demis
lname: Hassabis
motivation: "for protein structure prediction"
share: 4

year: 2024
category: Chemistry
fname: John
lname: Jumper
motivation: "for protein structure prediction"
share: 4

year: 2024
category: Economics
fname: Daron
lname: Acemoglu
motivation: "for studies of how institutions are formed and affect prosperity"
share: 3

year: 2024
category: Economics
fname: Simon
lname: Johnson
motivation: "for studies of how institutions are formed and affect prosperity"
share: 3

year: 2024
category: Economics
fname: James
lname: Robinson
motivation: "for studies of how institutions are formed and affect prosperity"
share: 3

year: 2024
category: Literature
fname: Kang
lname: Han
motivation: "for her intense poetic prose that confronts historical traumas and exposes the fragility of human life"
share: 1

year: 2024
category: Medici

### <font color="grey">Task 3.4.4: Create a list of distinct years where winners are listed</font>

In [17]:
# your code
cur.execute("""
SELECT DISTINCT year
FROM winners
WHERE year <= 2024
ORDER BY year
""")

years = [row[0] for row in cur.fetchall()]
years


[1901,
 1902,
 1903,
 1904,
 1905,
 1906,
 1907,
 1908,
 1909,
 1910,
 1911,
 1912,
 1913,
 1914,
 1915,
 1916,
 1917,
 1918,
 1919,
 1920,
 1921,
 1922,
 1923,
 1924,
 1925,
 1926,
 1927,
 1928,
 1929,
 1930,
 1931,
 1932,
 1933,
 1934,
 1935,
 1936,
 1937,
 1938,
 1939,
 1943,
 1944,
 1945,
 1946,
 1947,
 1948,
 1949,
 1950,
 1951,
 1952,
 1953,
 1954,
 1955,
 1956,
 1957,
 1958,
 1959,
 1960,
 1961,
 1962,
 1963,
 1964,
 1965,
 1966,
 1967,
 1968,
 1969,
 1970,
 1971,
 1972,
 1973,
 1974,
 1975,
 1976,
 1977,
 1978,
 1979,
 1980,
 1981,
 1982,
 1983,
 1984,
 1985,
 1986,
 1987,
 1988,
 1989,
 1990,
 1991,
 1992,
 1993,
 1994,
 1995,
 1996,
 1997,
 1998,
 1999,
 2000,
 2001,
 2002,
 2003,
 2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 2018,
 2019,
 2020,
 2021,
 2022,
 2023,
 2024]

### <font color="grey">For how many years are winners listed?</font>

In [18]:
# your code
cur.execute("""
SELECT COUNT(DISTINCT year)
FROM winners
WHERE year <= 2024
""")
cur.fetchone()[0]


121

**Interpretation question:** The Nobel Prize began in 1901. If the dataset runs through 2024, you would expect roughly 124 years of data - but the number of distinct years may be less.

Write code to identify which specific years are missing. Then explain why those years have no prize winners. (Hint: think about major world events.)

In [19]:
# Write code to identify missing years
cur.execute("""
SELECT DISTINCT year
FROM winners
WHERE year <= 2024
ORDER BY year
""")

years = [row[0] for row in cur.fetchall()]

all_years = set(range(1901, 2025))
missing_years = sorted(all_years - set(years))
missing_years


[1940, 1941, 1942]

<b>Write your answers below</b> <br>
*(Explanation of which years are missing and why)* <br>

The missing years are 1940, 1941, and 1942. These years were during World War II, which is why Nobel Prize winners were not listed for them. That explains why the total number of distinct years is 121 instead of the roughly 124 years from 1901 through 2024.


### <font color="grey">Task 3.4.5: Who are all the winners in 2020, and for what category?</font>

In [20]:
# your code
cur.execute("""
SELECT firstname, surname, category
FROM winners
WHERE year = 2020
ORDER BY category, surname
""")

rows = cur.fetchall()

for row in rows:
    print(f"{row[0]} {row[1]} - {row[2].title()}")


Emmanuelle Charpentier - Chemistry
Jennifer A. Doudna - Chemistry
Paul Milgrom - Economics
Robert Wilson - Economics
Louise Glück - Literature
Harvey Alter - Medicine
Michael Houghton - Medicine
Charles Rice - Medicine
World Food Programme  - Peace
Reinhard Genzel - Physics
Andrea Ghez - Physics
Roger Penrose - Physics


### <font color="grey">Task 3.4.6: Who were the winners of the Literature prize in the years 2010 thru 2020?</font>

In [21]:
# your code
cur.execute("""
SELECT year, firstname, surname
FROM winners
WHERE category = 'literature' AND year BETWEEN 2010 AND 2020
ORDER BY year
""")

rows = cur.fetchall()

for row in rows:
    print(f"{row[0]}: {row[1]} {row[2]}")


2010: Mario Vargas Llosa
2011: Tomas Tranströmer
2012: Mo Yan
2013: Alice Munro
2014: Patrick Modiano
2015: Svetlana Alexievich
2016: Bob Dylan
2017: Kazuo Ishiguro
2018: Olga Tokarczuk
2019: Peter Handke
2020: Louise Glück


### <font color="grey">Task 3.4.7: List all details of all the prizes that Einstein won</font>

In [22]:
# your code
cur.execute("""
SELECT *
FROM winners
WHERE surname = 'Einstein'
""")

rows = cur.fetchall()

for row in rows:
    print(row)


(1921, 'physics', 'Albert', 'Einstein', '"for his services to Theoretical Physics, and especially for his discovery of the law of the photoelectric effect"', 1)


### Reasoning Question 3.4.8 - Surprising Results

(a) How many prizes did Einstein win? Does this match what you expected?

(b) Many people associate Einstein with the Theory of Relativity, yet look at the `motivation` field in your result. What was he actually awarded for? Why do you think Relativity was not the reason?

(c) If you wanted to find all winners whose last name *contains* "stein" (not just Einstein), how would you modify your query? Write it below and run it.

<b>Write your answers below</b> <br>
*(a)* Einstein won 1 Nobel Prize. That may be surprising because people often connect him with many major discoveries, but he only won one Nobel Prize.

*(b)* He was awarded the prize for the discovery of the photoelectric effect, not for relativity. Relativity was probably not listed because it was still more controversial at the time and had not been accepted as clearly in the same way.

*(c)* To find all winners whose last name contains "stein", use LIKE with wildcards

In [23]:
# Query for all winners with 'stein' in their last name
cur.execute("""
SELECT *
FROM winners
WHERE surname LIKE '%stein%'
""")

cur.fetchall()


[(2011,
  'medicine',
  'Ralph M.',
  'Steinman',
  '"for his discovery of the dendritic cell and its role in adaptive immunity"',
  2),
 (1988,
  'physics',
  'Jack',
  'Steinberger',
  '"for the neutrino beam method and the demonstration of the doublet structure of the leptons through the discovery of the muon neutrino"',
  3),
 (1985,
  'medicine',
  'Joseph L.',
  'Goldstein',
  '"for their discoveries concerning the regulation of cholesterol metabolism"',
  2),
 (1984,
  'medicine',
  'César',
  'Milstein',
  '"for theories concerning the specificity in development and control of the immune system and the discovery of the principle for production of monoclonal antibodies"',
  3),
 (1972,
  'chemistry',
  'William H.',
  'Stein',
  '"for their contribution to the understanding of the connection between chemical structure and catalytic activity of the active centre of the ribonuclease molecule"',
  4),
 (1962,
  'literature',
  'John',
  'Steinbeck',
  '"for his realistic and imagin

## Problem 4 - Minimizing Redundancy of Data Storage

Minimizing redundancy is essential for efficient data management. Redundant data consumes unnecessary storage and creates consistency problems - if the same fact is stored in multiple places, they can fall out of sync. Techniques like normalization ensure each data point is stored once and referenced wherever needed.

### Exercise 4.0 - Identify Redundancy Before Measuring It

**Before writing any code:**

(a) In the `winners` table as currently designed, which fields do you expect to be repeated across multiple rows? List them and explain *why* they repeat.

(b) What is the difference between *necessary* repetition (data that must appear multiple times because it is genuinely different) and *redundant* repetition (the same information duplicated unnecessarily)?

(c) If we split the table into two tables to eliminate motivation redundancy, sketch what those two tables would look like. What columns goes in each? What links them together?

<b>Write your answers below</b> <br>
*(a)* In the winners table, the fields I would expect to repeat the most are year, category, and motivation. That is because one Nobel Prize in a given year and category can have multiple laureates, so those same prize details get repeated once for every winner tied to that prize. In some cases, even the share value can repeat when multiple laureates split the prize in the same way.

*(b)* Necessary repetition is when some data appears multiple times for a real reason, like different laureates belonging to the same prize. Redundant repetition is when the exact same information is copied again and again even though it could be stored once and referenced later. In this table, having multiple laureate rows for the same prize is necessary, but repeating the same prize motivation across those rows is redundant.

*(c) Sketch of two-table design:*
A cleaner design would split the table into one table for prize-level information and one table for laureate-level information.

**Table 1: prizes** ( ... )
prize_id, year, category, motivation

**Table 2: laureates** ( ... )
laureate_id, prize_id, firstname, surname, share

The two tables would be linked by prize_id, where prize_id is the primary key in prizes and a foreign key in laureates.

### <font color="grey">Task 4.1: Nobel prize shares</font>

How many total prizes were awarded? How many were shared prizes?

In [24]:
# your code
cur.execute("""
SELECT COUNT(DISTINCT year || '-' || category)
FROM winners
""")
cur.fetchone()[0]
# 633 prizes awarded

633

In [25]:
# your code
cur.execute("""
SELECT COUNT(*)
FROM (
    SELECT year, category
    FROM winners
    GROUP BY year, category
    HAVING COUNT(*) > 1
)
""")
cur.fetchone()[0]
# 271 prizes shared

271

**Interpretation:** What does the `share` field actually represent - is it the number of people sharing the prize, or the fraction each person receives? How can you tell from the data? Verify your interpretation with a specific example from the dataset.

*(Your interpretation and verification example)* <br>
The share field is the fraction of the prize each laureate receives, not the number of people sharing it. You can tell because the values do not always match the number of winners. For example, in the 2024 Chemistry prize, David Baker has share = 2, while Demis Hassabis and John Jumper each have share = 4. That means Baker received 1/2 of the prize, and Hassabis and Jumper each received 1/4, not that there were 2 or 4 people sharing it.

### <font color="grey">Task 4.2: Redundancy in the motivation</font>

There is redundancy in the motivation when a prize is shared. Find this redundancy and provide at least one example.

In [26]:
cur.execute("""
SELECT year, category, firstname, surname, motivation
FROM winners
WHERE year = 2024 AND category = 'physics'
""")

cur.fetchall()
# One example of redundancy is the 2024 Physics prize. The same motivation is repeated for both John Hopfield and Geoffrey Hinton. 
# Since they shared the same prize, the motivation text appears in more than one row even though it is really one prize-level detail.

[(2024,
  'physics',
  'John',
  'Hopfield',
  '"for foundational discoveries and inventions that enable machine learning with artificial neural networks"'),
 (2024,
  'physics',
  'Geoffrey',
  'Hinton',
  '"for foundational discoveries and inventions that enable machine learning with artificial neural networks"')]

### Exercise 4.2 - Consequences of Redundancy

(a) Suppose you discovered a typo in the motivation text for a shared prize (e.g., "protien" instead of "protein"). In the current single-table design, how many rows would you need to update to fix it?

(b) In your two-table design from Exercise 4.0(c), how many rows would you need to update? Why is this better?

(c) This principle has a formal name in database design. What is it called, and what "Normal Form" does it correspond to?

<b>Write your answers below</b> <br>
*(a)* In the current single-table design, I would need to update every row for that shared prize. So if the prize had 2 winners, I would update 2 rows. If it had 3 winners, I would update 3 rows.

*(b)* In the two-table design from 4.0(c), I would only need to update 1 row in the prizes table, because the motivation would be stored once per prize instead of repeated for every laureate. This is better because it reduces duplicate data and makes mistakes less likely.

*(c)* This is normalization, mainly moving toward Second Normal Form, because prize-level information like motivation should be stored separately instead of repeated across laureate rows.

### <font color="grey">Task 4.3: Retrieve</font>

Write a query to retrieve the year, first name, and last name from the winners table of all winners from 2019-2021 in the Physics category.

Sample output:
```
 2021 | Syukuro  | Manabe     
 2021 | Klaus    | Hasselmann 
 2021 | Giorgio  | Parisi     
 2020 | Roger    | Penrose    
 2020 | Reinhard | Genzel     
 2020 | Andrea   | Ghez       
 2019 | James    | Peebles    
 2019 | Michel   | Mayor      
 2019 | Didier   | Queloz     
```

**Before coding:** There are at least two different SQL approaches to filter for years 2019-2021. Identify both approaches, state which you will use, and explain why.

<b>Write your answers below</b> <br>
*(Two approaches and your reasoning for choosing one)* <br>
Two SQL approaches to filter for 2019–2021 are year BETWEEN 2019 AND 2021 and year >= 2019 AND year <= 2021. Another possible way is year IN (2019, 2020, 2021). I would use BETWEEN because it is shorter and clearly shows that I want an inclusive range.


In [27]:
# your code
cur.execute("""
SELECT year, firstname, surname
FROM winners
WHERE year BETWEEN 2019 AND 2021
  AND category = 'physics'
ORDER BY year DESC
""")

rows = cur.fetchall()

for row in rows:
    print(f"{row[0]} | {row[1]} | {row[2]}")


2021 | Syukuro | Manabe
2021 | Klaus | Hasselmann
2021 | Giorgio | Parisi
2020 | Roger | Penrose
2020 | Reinhard | Genzel
2020 | Andrea | Ghez
2019 | James | Peebles
2019 | Michel | Mayor
2019 | Didier | Queloz


### Exercise 4.4 - Open-Ended: Your Own Analytical Query

Write one query of your own design that reveals something **interesting or surprising** about the Nobel Prize data. It must use at least one of: `GROUP BY`, `ORDER BY`, `HAVING`, or a subquery.

(a) State what question you are investigating and why you find it interesting.

(b) Write and run the query.

(c) Interpret the result in 2-3 sentences. What does it tell you about Nobel Prizes?

<b>Write your answers below</b> <br>
*(a) My question:* Which Nobel laureates won more than one prize? I think this is worth knowing because most people only win once, so it would be surprising to see who appears more than once.

In [28]:
# (b) your query
cur.execute("""
SELECT firstname, surname, COUNT(*) AS prizes_won
FROM winners
GROUP BY firstname, surname
HAVING COUNT(*) > 1
ORDER BY prizes_won DESC, surname, firstname
""")

rows = cur.fetchall()

for row in rows:
    print(f"{row[0]} {row[1]} - {row[2]}")


International Committee of the Red Cross  - 3
Office of the United Nations High Commissioner for Refugees  - 2
John Bardeen - 2
Marie Curie - 2
Linus Pauling - 2
Frederick Sanger - 2
Barry Sharpless - 2


*(c) Interpretation:* <br>
This result shows that winning more than one Nobel Prize is very rare. Only a small number of names appear more than once, and some of them are organizations, not just individual people. That makes the repeat winners stand out and shows that Nobel Prizes usually go to different laureates rather than the same ones over and over.

### Close the database connection

In [29]:
conn.close()

## Final Reflection

Answer the following in your own words (4-6 sentences total):

(a) What is the most important concept you reinforced or learned in this lab?

(b) Describe one decision you made where there was more than one valid choice (e.g., data type selection, query approach, schema design). What did you choose and why?

(c) If you were designing a full Nobel Prize web application for millions of users, what would you change about the single-table database design used in this lab?

<b>Write your answers below</b> <br>
*(a)* The main thing I learned from this lab was that database design really matters. A table can work at first, but if it stores too much repeated information, it becomes harder to update and manage. I also got more comfortable with turning JSON data into tables and using SQL to query it.

*(b)* One decision I had to make was how to identify rows uniquely in the winners table. At first, something like year and category seemed like it could work, but that would not be unique because multiple laureates can share the same prize. I chose the surrogate key approach because it gives each row its own unique identifier and makes the design safer.

*(c)* For a real Nobel Prize app with lots of users, I would split the data into multiple tables instead of keeping it all in one. That would reduce repeated information, make updates easier, and improve performance. I would also use indexes so searches and filters run faster.

### Submission Instructions
Rename the lab as netID_Lab04.ipynb and submit to codebench with the log file. (eg: adg133_Lab04.ipynb)